In [7]:
import torch
import torch.nn as nn
import torch.onnx as onnx

In [8]:
model_path = "simple_linear_model.onnx"

In [9]:
# Create a simple one layer model using a linear layer


class SimpleLinearModel(nn.Module):
    def __init__(self, input_size, output_size):
        super(SimpleLinearModel, self).__init__()
        # Define a single linear layer
        self.linear = nn.Linear(input_size, 10)
        self.linear2 = nn.Linear(10, 20)
        self.linear3 = nn.Linear(20, 15)
        self.linear4 = nn.Linear(15, output_size)

    def forward(self, x):
        # Pass input through the linear layer
        output = self.linear(x)
        output = self.linear2(output)
        output = self.linear3(output)
        output = self.linear4(output)
        return output


# Example usage
model = SimpleLinearModel(input_size=10, output_size=5)
print(model)

print("Model weights:")
for name, param in model.named_parameters():
    print(f"{name}: {param.shape}")
    if "weight" in name:
        print(f"  Weight values (first 5): {param.flatten()[:5]}")
    elif "bias" in name:
        print(f"  Bias values (first 5): {param.flatten()[:5]}")

SimpleLinearModel(
  (linear): Linear(in_features=10, out_features=10, bias=True)
  (linear2): Linear(in_features=10, out_features=20, bias=True)
  (linear3): Linear(in_features=20, out_features=15, bias=True)
  (linear4): Linear(in_features=15, out_features=5, bias=True)
)
Model weights:
linear.weight: torch.Size([10, 10])
  Weight values (first 5): tensor([ 0.2052, -0.1498,  0.1389,  0.2674, -0.0629], grad_fn=<SliceBackward0>)
linear.bias: torch.Size([10])
  Bias values (first 5): tensor([ 0.0494,  0.1764, -0.1545, -0.1388,  0.1485], grad_fn=<SliceBackward0>)
linear2.weight: torch.Size([20, 10])
  Weight values (first 5): tensor([ 0.2880,  0.0528,  0.1274, -0.1425,  0.0239], grad_fn=<SliceBackward0>)
linear2.bias: torch.Size([20])
  Bias values (first 5): tensor([-0.0563, -0.1781,  0.3033, -0.1253,  0.0131], grad_fn=<SliceBackward0>)
linear3.weight: torch.Size([15, 20])
  Weight values (first 5): tensor([-0.1583, -0.2151, -0.0436,  0.1943, -0.1123], grad_fn=<SliceBackward0>)
linear3.

In [10]:
# export to onnx
onnx.export(model, torch.randn(1, 10), model_path, export_params=True, opset_version=11)

In [11]:
# run the model with pytorch
input_data = torch.tensor([[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]])
with torch.no_grad():
    output = model(input_data)
print(output)

tensor([[-8.9779e-06, -2.8664e-01,  1.4374e-01,  1.2125e-01, -2.4952e-01]])


In [12]:
from c_exporter.onnx_exporter import export_onnx

output_model = export_onnx(model_path)
print(output_model)

{'input_size': 10, 'output_size': 5, 'layers': [{'type': 'dense', 'input_size': 10, 'output_size': 10, 'weights': [0.20520329475402832, -0.14983819425106049, 0.138935387134552, 0.2674339711666107, -0.06292246282100677, -0.3016625642776489, -0.24117770791053772, 0.1952551007270813, -0.026482420042157173, -0.2152501940727234, 0.08784851431846619, -0.2828574776649475, 0.23182930052280426, 0.11746746301651001, -0.1702413558959961, 0.27650877833366394, -0.18439364433288574, -0.1132182627916336, 0.23091654479503632, -0.11735361814498901, -0.28453174233436584, 0.2813820540904999, 0.04331248626112938, 0.2540189027786255, 0.24228359758853912, -0.13354064524173737, 0.12156541645526886, -0.09910541027784348, 0.031010428443551064, -0.17501749098300934, 0.2141977995634079, 0.1160685122013092, -0.27590787410736084, -0.30219411849975586, -0.16290083527565002, 0.3019581735134125, -0.15762972831726074, -0.22239427268505096, -0.02416580729186535, 0.27684345841407776, -0.1924048364162445, 0.3019779622554